# Gradient Boosting — Full Mathematical Derivation

## 1. The Additive Model

Gradient Boosting builds the final model as a **sum of functions**, added one at a time:

$$f(x) = f_0(x) + f_1(x) + f_2(x) + \dots + f_M(x)$$

- $f_0(x)$ — the initial (base) model, just a constant.
- $f_1(x), f_2(x), \dots$ — successive trees, each correcting the errors of the model built so far.
- $M$ — total number of boosting rounds (trees).

---

## 2. Step 1: Initialize the Model — Find $f_0(x)$

$f_0(x)$ is a **constant** $\gamma$ that minimizes the total loss over all training points:

$$f_0(x) = \arg\min_{\gamma} \sum_{i=1}^{n} L(y_i, \gamma)$$

Using squared error loss:

$$L = \frac{1}{2}\sum_{i=1}^{n} (y_i - \gamma)^2$$

$$f_0(x) = \arg\min_{\gamma} \frac{1}{2}\sum_{i=1}^{n} (y_i - \gamma)^2$$

### Solving for $\gamma$ (setting derivative to 0)

$$\frac{d}{d\gamma}\sum_{i=1}^n (y_i - \gamma)^2 = -\sum_{i=1}^n (y_i - \gamma) = 0$$

$$\sum_{i=1}^{n} (\gamma - y_i) = 0$$

Expanding for n points (e.g. n = 3 in the board example):

$$(\gamma - y_1) + (\gamma - y_2) + (\gamma - y_3) = 0$$

Solving this gives:

$$\gamma = \frac{1}{n}\sum_{i=1}^n y_i$$

**→ $f_0(x)$ is simply the mean of all target values $y_i$.** This is the same result used in the earlier salary example (`pred1 = 4.8`, the mean).

---

## 3. Step 2: Compute Pseudo-Residuals

For each boosting round $m$, and each data point $i$, the pseudo-residual is defined as the **negative gradient of the loss function** with respect to the current prediction $\hat{y}_i$:

$$r_{im} = -\left[\frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i}\right]_{f = f_{m-1}}$$

### Why this equals $(y_i - \hat{y}_i)$ for squared-error loss

With $L = \frac{1}{2}(y_i - \hat{y}_i)^2$:

$$-\frac{\partial}{\partial \hat{y}_i}\left[\frac{1}{2}(y_i - \hat{y}_i)^2\right] = (y_i - \hat{y}_i)$$

So the pseudo-residual is exactly the ordinary residual:

$$r_{i1} = y_i - f_0(x_i)$$

**Example from the board:**
$$r_{31} = y_3 - f_0(x_3) = 91 - 142 = -51$$

This confirms: gradient boosting's "residual fitting" is a special case of gradient descent in function space — at each step we move in the direction that most reduces the loss.

---

## 4. Step 3: Fit a Tree to the Residuals

- A regression tree is trained with the **pseudo-residuals $r_{im}$** as the target (not the original $y_i$).
- This partitions the input space (based on features like $x$) into **terminal regions (leaves)**:

$$R_{jm}, \quad j = 1, 2, \dots, J_m$$

where $J_m$ = number of leaves in tree $m$.

Each region $R_{jm}$ groups together data points that fall into the same leaf — visualized on the board as splitting the feature space (`x | y`) into partitions like $R_{11}, R_{21}$, etc., corresponding to leaves of the tree ($m = 1$ → first tree).

---

## 5. Step 4: Compute the Optimal Leaf Value $\gamma_{jm}$

For **each leaf/region** $R_{jm}$, we don't just average the residuals — we solve a small optimization to find the value $\gamma_{jm}$ that minimizes the **original loss function** (not just the residual-fitting error) for the points in that region:

$$\gamma_{jm} = \arg\min_{\gamma} \sum_{x_i \in R_{jm}} L\big(y_i,\ f_{m-1}(x_i) + \gamma\big)$$

- For squared-error loss, this simplifies to the **mean residual** in that region.
- For other losses (e.g. absolute error, log-loss for classification), this step can give a different value than the plain average — this is why it's written as an explicit argmin rather than just "average the residuals."

---

## 6. Step 5: Update the Model

Once every leaf's optimal value $\gamma_{jm}$ is found, update the overall model:

$$f_m(x) = f_{m-1}(x) + \sum_{j=1}^{J_m} \gamma_{jm}\, I(x \in R_{jm})$$

Here $I(x \in R_{jm})$ is an indicator function — it's 1 if $x$ falls in region $R_{jm}$, else 0. So effectively, each point gets the $\gamma_{jm}$ value of whichever leaf it lands in, added on top of the previous prediction.

### Sequential build-up
$$f_1(x) = f_0(x) + \gamma_{\cdot 1}\ (\text{tree 1 output})$$
$$f_2(x) = f_1(x) + \gamma_{\cdot 2}\ (\text{tree 2 output})$$
$$\vdots$$

(In practice a **learning rate** $\eta$ is multiplied into each tree's contribution to control overfitting: $f_m(x) = f_{m-1}(x) + \eta \sum \gamma_{jm} I(x \in R_{jm})$.)

---

## 7. Final Output

After $M$ boosting rounds:

$$\hat{f}(x) = f_M(x)$$

---

## Algorithm Summary (Full Loop)

1. **Initialize**: $f_0(x) = \arg\min_\gamma \sum_i L(y_i, \gamma)$ → mean of $y$ for squared error.
2. **For** $m = 1$ to $M$:
   - (a) Compute pseudo-residuals: $r_{im} = -\left[\frac{\partial L(y_i,\hat y_i)}{\partial \hat y_i}\right]_{f=f_{m-1}}$
   - (b) Fit a regression tree to targets $r_{im}$, giving terminal regions $R_{jm}$, $j=1,\dots,J_m$.
   - (c) For each region, compute $\gamma_{jm} = \arg\min_\gamma \sum_{x_i \in R_{jm}} L(y_i, f_{m-1}(x_i)+\gamma)$.
   - (d) Update: $f_m(x) = f_{m-1}(x) + \sum_{j=1}^{J_m}\gamma_{jm} I(x \in R_{jm})$.
3. **Output**: $\hat f(x) = f_M(x)$.

---

## Key Takeaways
- Gradient Boosting = **gradient descent in function space**, where each "step" is a tree instead of a parameter update.
- Squared-error loss makes the pseudo-residual literally equal to $(y - \hat{y})$, which is why the simplified explanation ("fit trees to residuals") works — but the general framework (via $\gamma_{jm}$'s argmin) supports **any differentiable loss function**.
- $f_0(x)$ = mean of targets (for regression w/ squared error).
- Leaf values $\gamma_{jm}$ are computed by minimizing the actual loss per region, not just averaged blindly — important for non-squared losses.
- Learning rate shrinks each tree's contribution to reduce overfitting.